## Exercise 2

In [119]:
import pandas as pd

from sklearn import datasets
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor
from sklearn.tree import ExtraTreeRegressor
from sklearn.metrics import mean_squared_error, r2_score

import matplotlib.pyplot as plt
import numpy as np


# from sklearn.model_selection import train_test_split
from sklearn.pipeline import make_pipeline
# from sklearn.linear_model import LinearRegression
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.cross_decomposition import PLSRegression

### 2.1 Import Outage Data

In [120]:
%%time

csvPath = "C:/Users/TJones/OneDrive/Documents/IEEE/Presentations/2026 T&D/2026 TD Tutorial/Tyler's presentation material/outage_data.csv"
df = pd.read_csv(csvPath)

# print(df)
print(df.columns.tolist())

display(df)

['outage_id', 'phase', 'year', 'outage_start', 'outage_end', 'duration_min', 'ci', 'cmi', 'cause_category', 'customers', 'day', 'saidi', 'saifi', 'caidi']


,outage_id,phase,year,outage_start,outage_end,duration_min,ci,cmi,cause_category,customers,day,saidi,saifi,caidi
0,1367575,C,2014,2014-01-01 00:10:30,2014-01-01 04:20:24,249.9000,1,249.900,Equipment,2552822,2014-01-01,0.000098,0.0000,249.900000
1,1359015,B,2014,2014-01-01 00:15:12,2014-01-01 01:06:53,51.6833,7,361.783,Wildlife,2552822,2014-01-01,0.000142,0.0000,51.683286
2,1377206,B,2014,2014-01-01 00:34:11,2014-01-01 09:32:12,538.0167,1,538.017,Equipment,2552822,2014-01-01,0.000211,0.0000,538.017000
3,1378722,A,2014,2014-01-01 00:50:14,2014-01-01 01:40:14,50.0000,3,150.000,Other,2552822,2014-01-01,0.000059,0.0000,50.000000
4,1362829,AB,2014,2014-01-01 01:56:39,2014-01-01 02:46:34,49.9167,141,7038.250,Planned,2552822,2014-01-01,0.002757,0.0001,49.916667
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
70441,1863521,C,2018,2018-12-31 22:21:12,2019-01-01 01:10:20,169.1333,1,169.133,Equipment,2555928,2018-12-31,0.000066,0.0000,169.133000
70442,1861169,C,2018,2018-12-31 22:40:11,2019-01-01 03:53:55,313.7333,4,1254.933,Equipment,2555928,2018-12-31,0.000491,0.0000,313.733250
70443,1860009,ABC,2018,2018-12-31 23:16:35,2019-01-01 01:56:26,159.8500,1,159.850,Equipment,2555928,2018-12-31,0.000063,0.0000,159.850000
70444,1860296,B,2018,2018-12-31 23:16:58,2019-01-01 00:32:32,75.5667,1,75.567,Other,2555928,2018-12-31,0.000030,0.0000,75.567000


CPU times: total: 141 ms
Wall time: 129 ms


### 2.2 Calculate IEEE TMED

1. sum CI and SAIDI by day
2. filter where sum CI > 0
3. add the ln(SAIDI) column
4. calculate Alpha and Beta
5. calculate TMED


In [ ]:
daily = (
  df.groupby('day', as_index=False)
    .agg(
      outage_count=('outage_id', 'nunique'),
      ci=('ci', 'sum'),
      cmi=('cmi', 'sum'),
      saidi=('saidi', 'sum'),
      saifi=('saifi', 'sum'),
      caidi=('caidi', 'sum'),
      duration_min=('duration_min', 'sum'),
      customers=('customers', 'max')
    )
    .sort_values('day')
)

# full_days = pd.date_range(daily['day'].min(), daily['day'].max(), freq='D')
# daily = daily.set_index('day').reindex(full_days).rename_axis('day').reset_index()
daily['weekday'] = daily['day'].dt.day_name()

daily['year'] = daily['day'].dt.year
daily['month'] = daily['day'].dt.month
daily['weekday'] = daily['day'].dt.day_name()

alpha = np.log(daily.loc[daily["saidi"] > 0, "saidi"]).mean()
beta = np.log(daily.loc[daily["saidi"] > 0, "saidi"]).std()

tmed = np.exp(alpha + 2.5 * beta)

print(f"Alpha: {alpha:.5f}")
print(f"Beta :  {beta:.5f}")
print(f"TMED :  {tmed:.2f}")

display(daily)

Alpha: nan
Beta :  nan
TMED :  nan


,day,outage_count,ci,cmi,saidi,saifi,caidi,duration_min,customers,weekday,year,month
0,2014-01-01,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Wednesday,2014,1
1,2014-01-02,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Thursday,2014,1
2,2014-01-03,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Friday,2014,1
3,2014-01-04,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Saturday,2014,1
4,2014-01-05,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Sunday,2014,1
...,...,...,...,...,...,...,...,...,...,...,...,...
1821,2018-12-27,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Thursday,2018,12
1822,2018-12-28,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Friday,2018,12
1823,2018-12-29,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Saturday,2018,12
1824,2018-12-30,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Sunday,2018,12


In [ ]:
# Month x weekday heatmap by outage count.
weekday_order = ['Monday', 'Tuesday', 'Wednesday', 'Thursday', 'Friday', 'Saturday', 'Sunday']
heat = (
  daily.pivot_table(index='month', columns='weekday', values='outage_count', aggfunc='sum')
    .reindex(columns=weekday_order)
    .fillna(0)
)

fig, ax = plt.subplots(figsize=(11, 6))
im = ax.imshow(heat.values, aspect='auto')
ax.set_title('Seasonality Heatmap — Outage Count by Month and Weekday')
ax.set_xlabel('Weekday')
ax.set_ylabel('Month')
ax.set_xticks(np.arange(len(heat.columns)))
ax.set_xticklabels(heat.columns, rotation=45, ha='right')
ax.set_yticks(np.arange(len(heat.index)))
ax.set_yticklabels(heat.index)
fig.colorbar(im, ax=ax, label='Outage count')
fig.tight_layout()
plt.show()

### 2.3 Identify a Major Event

In [ ]:
# using the "daily" data frame from 1.2
major_event_days = daily[daily["saidi"] >= tmed].copy()
# append tmed so it can be seen
major_event_days["tmed"] = tmed

print(major_event_days)

### 2.4 MED Intervals

In [ ]:
df["outage_start"] = pd.to_datetime(df["outage_start"])
df["outage_end"] = pd.to_datetime(df["outage_end"])
df["day"] = df["outage_start"].dt.floor("D")

major_event_days["day"] = pd.to_datetime(major_event_days["day"]).dt.floor("D")

for event_day in sorted(major_event_days["day"].dropna().unique()):
  event_day = pd.Timestamp(event_day)

  start = event_day - pd.Timedelta(days=1)
  end = event_day + pd.Timedelta(days=1)

  event_window = df[
    (df["outage_start"] >= start) &
    (df["outage_start"] < end)
  ].copy()

  if event_window.empty:
    continue

  event_window = event_window.sort_values("outage_start")
  event_window["cumulative_saidi"] = event_window["saidi"].cumsum()

  starts = event_window[["outage_start"]].rename(columns={"outage_start": "timestamp"})
  starts["delta"] = 1

  ends = event_window[["outage_end"]].rename(columns={"outage_end": "timestamp"})
  ends["delta"] = -1

  outage_count = (
    pd.concat([starts, ends], ignore_index=True)
    .dropna(subset=["timestamp"])
    .sort_values("timestamp")
  )

  outage_count["active_outage_count"] = outage_count["delta"].cumsum()

  fig, ax1 = plt.subplots(figsize=(12, 5))

  ax1.plot(
    event_window["outage_start"],
    event_window["cumulative_saidi"],
    marker="o",
    linewidth=1,
    label="Cumulative SAIDI"
  )

  ax1.axhline(
    y=tmed,
    linestyle="--",
    linewidth=2,
    label=f"TMED = {tmed:.2f}"
  )

  ax1.set_title(f"Cumulative SAIDI Around Major Event Day: {event_day.date()}")
  ax1.set_xlabel("Time")
  ax1.set_ylabel("Cumulative SAIDI")
  ax1.grid(True, alpha=0.3)

  ax2 = ax1.twinx()

  ax2.step(
    outage_count["timestamp"],
    outage_count["active_outage_count"],
    where="post",
    linewidth=1,
    label="Active Outage Count"
  )

  ax2.set_ylabel("Active Outage Count")

  lines1, labels1 = ax1.get_legend_handles_labels()
  lines2, labels2 = ax2.get_legend_handles_labels()
  ax1.legend(lines1 + lines2, labels1 + labels2, loc="upper left")

  plt.tight_layout()
  plt.show()

### 2.5 Rolling 365 Charts

In [ ]:
df["outage_start"] = pd.to_datetime(df["outage_start"])
df["day"] = df["outage_start"].dt.floor("D")

major_event_days["day"] = pd.to_datetime(major_event_days["day"]).dt.floor("D")

df = df.sort_values("outage_start").copy()

df["rolling_365_saidi"] = (
  df["saidi"]
    .rolling(window=365, min_periods=365)
    .sum()
)

df_excl_med = (
  df[~df["day"].isin(major_event_days["day"])]
    .sort_values("outage_start")
    .copy()
)

df_excl_med["rolling_365_saidi_excl_med"] = (
  df_excl_med["saidi"]
    .rolling(window=365, min_periods=365)
    .sum()
)

plt.figure(figsize=(12, 5))

plt.plot(
  df["outage_start"],
  df["rolling_365_saidi"],
  linewidth=1,
  label="Rolling 365 SAIDI"
)

plt.plot(
  df_excl_med["outage_start"],
  df_excl_med["rolling_365_saidi_excl_med"],
  linewidth=1,
  label="Rolling 365 SAIDI Excluding MEDs"
)

plt.title("Rolling 365-Row SAIDI With and Without Major Event Days")
plt.xlabel("Outage Start")
plt.ylabel("Rolling 365 SAIDI")
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

### 2.6 Year-over-Year

In [ ]:
annual = (
  df.groupby("year", as_index=False)
    .agg(
      saidi=("saidi", "sum"),
      saifi=("saifi", "sum")
    )
)

annual["caidi"] = annual["saidi"] / annual["saifi"]

fig, ax1 = plt.subplots(figsize=(12, 5))

ax1.plot(
  annual["year"],
  annual["saidi"],
  marker="o",
  linewidth=2,
  label="SAIDI"
)

ax1.plot(
  annual["year"],
  annual["caidi"],
  marker="o",
  linewidth=2,
  label="CAIDI"
)

ax1.set_xlabel("Year")
ax1.set_ylabel("SAIDI / CAIDI")
ax1.grid(True, alpha=0.3)

ax2 = ax1.twinx()

ax2.plot(
  annual["year"],
  annual["saifi"],
  marker="x",
  linewidth=2,
  label="SAIFI"
)

ax2.set_ylabel("SAIFI")

lines1, labels1 = ax1.get_legend_handles_labels()
lines2, labels2 = ax2.get_legend_handles_labels()

ax1.legend(lines1 + lines2, labels1 + labels2, loc="upper left")

plt.title("Annual SAIDI, SAIFI, and CAIDI")
plt.tight_layout()
plt.show()

### 2.7 Cause Analysis

In [ ]:
cause = (
  df.groupby('cause_category', as_index=False)
    .agg(
      outage_count=('outage_id', 'nunique'),
      ci=('ci', 'sum'),
      cmi=('cmi', 'sum'),
      duration_min=('duration_min', 'sum'),
      avg_ci=('ci', 'mean'),
      avg_cmi=('cmi', 'mean'),
      avg_duration_min=('duration_min', 'mean')
    )
    .sort_values('cmi', ascending=False)
)

cause['cmi_share_pct'] = cause['cmi'] / cause['cmi'].sum() * 100
cause['ci_share_pct'] = cause['ci'] / cause['ci'].sum() * 100
cause['outage_share_pct'] = cause['outage_count'] / cause['outage_count'].sum() * 100
cause['caidi'] = np.where(cause['ci'] > 0, cause['cmi'] / cause['ci'], np.nan)
cause['cumulative_cmi_share_pct'] = cause['cmi_share_pct'].cumsum()

display(cause)

In [ ]:
# Cause Pareto by CMI.
fig, ax1 = plt.subplots(figsize=(12, 5))
plot_cause = cause.head(15).copy()
ax1.bar(plot_cause['cause_category'], plot_cause['cmi'])
ax1.set_ylabel('CMI')
ax1.set_xlabel('Cause')
ax1.tick_params(axis='x', rotation=45)

ax2 = ax1.twinx()
ax2.plot(plot_cause['cause_category'], plot_cause['cumulative_cmi_share_pct'], marker='o')
ax2.set_ylabel('Cumulative CMI share %')
ax1.set_title('Cause Pareto — CMI Impact')
fig.tight_layout()
plt.show()

# Cause severity matrix: frequency vs average CMI.
fig, ax = plt.subplots(figsize=(9, 6))
ax.scatter(cause['outage_count'], cause['avg_cmi'])
for _, r in cause.head(12).iterrows():
  ax.annotate(r['cause_category'], (r['outage_count'], r['avg_cmi']), fontsize=8, xytext=(4, 4), textcoords='offset points')
ax.set_title('Cause Severity Matrix — Frequency vs Average CMI')
ax.set_xlabel('Outage count')
ax.set_ylabel('Average CMI per outage')
ax.grid(True, alpha=0.3)
fig.tight_layout()
plt.show()

In [ ]:
# Cause mix by year using CMI share.
cause_year = (
  df.groupby(['year', 'cause_category'], as_index=False)
  .agg(cmi=('cmi', 'sum'), ci=('ci', 'sum'), outage_count=('outage_id', 'nunique'))
)

top_causes = cause.head(8)['cause_category'].tolist()
cause_year['cause_group'] = np.where(cause_year['cause_category'].isin(top_causes), cause_year['cause_category'], 'Other causes')

cause_year_grouped = (
  cause_year.groupby(['year', 'cause_group'], as_index=False).agg(cmi=('cmi', 'sum'), ci=('ci', 'sum'), outage_count=('outage_count', 'sum'))
)

pivot_cmi = cause_year_grouped.pivot(index='year', columns='cause_group', values='cmi').fillna(0)
pivot_share = pivot_cmi.div(pivot_cmi.sum(axis=1).replace(0, np.nan), axis=0) * 100

ax = pivot_share.plot(kind='bar', stacked=True, figsize=(12, 6))
ax.set_title('Year-over-Year Cause Mix by CMI Share')
ax.set_xlabel('Year')
ax.set_ylabel('CMI share %')
ax.legend(title='Cause', bbox_to_anchor=(1.02, 1), loc='upper left')
plt.tight_layout()
plt.show()